# SVM Model Guide:-

- BigQuery ML does not have a native model_type = 'svm' or model_type = 'svm_classifier'.
- In production and GCP environments, the standard alternatives for datasets where SVM is traditionally used are:
    1. **Linear SVM Alternative**: LOGISTIC_REG with L2 Regularization (l2_reg) — performs the exact same task of finding the optimal linear separating boundary/margin.
    2. **Kernel (Non-linear / RBF) SVM Alternative**: DNN_CLASSIFIER (Deep Neural Network) or BOOSTED_TREE_CLASSIFIER — creates non-linear decision boundaries.

- Below is the complete BigQuery ML code for the Breast Cancer Wisconsin (breast_cancer.csv) dataset, using the Linear Margin Classifier (LOGISTIC_REG + L2 Regularization) as the direct BigQuery equivalent to Linear SVM, along with an optional Deep Neural Network (Kernel SVM equivalent).

# Complete Step by Step Guide for SVM Model:-

- **Step 1**: Create Schema / Dataset
```sql 
-- Create a schema/dataset in the US region
CREATE SCHEMA IF NOT EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox`
OPTIONS(
  location = 'US'
);
```

---

> gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_breast_cancer

- **Step 2**: Cleanse and Preprocess Data -> In the Breast Cancer dataset:
    - Drop id and any trailing empty column (like Unnamed: 32).
    - Ensure diagnosis is formatted as 'M' (Malignant) vs 'B' (Benign) or 1 vs 0.

```sql 
-- Cleanse data: exclude ID and clean diagnosis column
CREATE OR REPLACE TABLE `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_breast_cancer` AS
SELECT
  -- Target column: 'M' (Malignant) or 'B' (Benign)
  CAST(diagnosis AS STRING) AS diagnosis,
  
  -- Mean Features
  CAST(radius_mean AS FLOAT64) AS radius_mean,
  CAST(texture_mean AS FLOAT64) AS texture_mean,
  CAST(perimeter_mean AS FLOAT64) AS perimeter_mean,
  CAST(area_mean AS FLOAT64) AS area_mean,
  CAST(smoothness_mean AS FLOAT64) AS smoothness_mean,
  CAST(compactness_mean AS FLOAT64) AS compactness_mean,
  CAST(concavity_mean AS FLOAT64) AS concavity_mean,
  CAST(`concave points_mean` AS FLOAT64) AS concave_points_mean,
  CAST(symmetry_mean AS FLOAT64) AS symmetry_mean,
  CAST(fractal_dimension_mean AS FLOAT64) AS fractal_dimension_mean,
  
  -- Standard Error Features
  CAST(radius_se AS FLOAT64) AS radius_se,
  CAST(texture_se AS FLOAT64) AS texture_se,
  CAST(perimeter_se AS FLOAT64) AS perimeter_se,
  CAST(area_se AS FLOAT64) AS area_se,
  CAST(smoothness_se AS FLOAT64) AS smoothness_se,
  CAST(compactness_se AS FLOAT64) AS compactness_se,
  CAST(concavity_se AS FLOAT64) AS concavity_se,
  CAST(`concave points_se` AS FLOAT64) AS concave_points_se,
  CAST(symmetry_se AS FLOAT64) AS symmetry_se,
  CAST(fractal_dimension_se AS FLOAT64) AS fractal_dimension_se,
  
  -- Worst / Largest Features
  CAST(radius_worst AS FLOAT64) AS radius_worst,
  CAST(texture_worst AS FLOAT64) AS texture_worst,
  CAST(perimeter_worst AS FLOAT64) AS perimeter_worst,
  CAST(area_worst AS FLOAT64) AS area_worst,
  CAST(smoothness_worst AS FLOAT64) AS smoothness_worst,
  CAST(compactness_worst AS FLOAT64) AS compactness_worst,
  CAST(concavity_worst AS FLOAT64) AS concavity_worst,
  CAST(`concave points_worst` AS FLOAT64) AS concave_points_worst,
  CAST(symmetry_worst AS FLOAT64) AS symmetry_worst,
  CAST(fractal_dimension_worst AS FLOAT64) AS fractal_dimension_worst
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_breast_cancer`;
```

---

- **Step 3**: Train the Linear Classifier (Linear SVM Equivalent) -> Uses strong L2 Regularization (l2_reg = 1.0) to maximize the separating margin between Benign and Malignant tumors.

```sql 
-- Train Linear Maximum-Margin Classifier (Linear SVM equivalent)
CREATE OR REPLACE MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.breast_cancer_model`
OPTIONS (
  model_type = 'LOGISTIC_REG',
  input_label_cols = ['diagnosis'],
  
  -- L2 Regularization (Penalizes large weights like an SVM C-parameter)
  l2_reg = 1.0,
  
  -- 80% train / 20% test split
  data_split_method = 'RANDOM',
  data_split_eval_fraction = 0.2,
  auto_class_weights = TRUE
) AS
SELECT
  *
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_breast_cancer`;
```

- Non-Linear / Kernel SVM Equivalent (Deep Neural Network):

```sql
-- If you need non-linear decision boundaries (Kernel SVM equivalent):
CREATE OR REPLACE MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.breast_cancer_dnn_model`
OPTIONS (
  model_type = 'DNN_CLASSIFIER',
  input_label_cols = ['diagnosis'],
  hidden_units = [32, 16],
  dropout = 0.2,
  data_split_method = 'RANDOM',
  data_split_eval_fraction = 0.2
) AS
SELECT * FROM `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_breast_cancer`;
```

---


- **Step 4**: Evaluate Model Performance -> Check **Accuracy**, **Precision**, **Recall**, and **ROC AUC** (Cancer diagnosis models require high recall to avoid missing positive cancer cases).

```sql 
-- Evaluate classification performance
SELECT
  *
FROM
  ML.EVALUATE(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.breast_cancer_model`);
```

---


- **Step 5**: Confusion Matrix -> Crucial for medical diagnosis to check for False Negatives (malignant tumors falsely predicted as benign).

```sql 
-- Evaluate Confusion Matrix
SELECT
  *
FROM
  ML.CONFUSION_MATRIX(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.breast_cancer_model`);
```

---


- **Step 6**: View Feature Weights (Margin Separation) -> 
See which cell nuclei characteristics are the strongest indicators of malignancy (e.g., perimeter_worst, concave_points_worst).

```sql 
-- Check feature weights
SELECT
  processed_input,
  weight
FROM
  ML.WEIGHTS(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.breast_cancer_model`)
ORDER BY
  ABS(weight) DESC;
```

---


- **Step 7**: Predict on Unseen Biopsy Samples -> Predict whether a tumor is Benign (B) or Malignant (M) with probability scores:

```sql 
-- Make predictions on test biopsy measurements
SELECT
  predicted_diagnosis,
  predicted_diagnosis_probs,
  radius_mean,
  texture_mean,
  area_mean,
  concavity_mean
FROM
  ML.PREDICT(
    MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.breast_cancer_model`,
    (
      -- Sample 1: Benign biopsy profile (smaller radius, smooth margins)
      SELECT
        CAST(13.54 AS FLOAT64) AS radius_mean, CAST(14.36 AS FLOAT64) AS texture_mean,
        CAST(87.46 AS FLOAT64) AS perimeter_mean, CAST(566.3 AS FLOAT64) AS area_mean,
        CAST(0.09779 AS FLOAT64) AS smoothness_mean, CAST(0.08129 AS FLOAT64) AS compactness_mean,
        CAST(0.06664 AS FLOAT64) AS concavity_mean, CAST(0.04781 AS FLOAT64) AS concave_points_mean,
        CAST(0.1885 AS FLOAT64) AS symmetry_mean, CAST(0.05766 AS FLOAT64) AS fractal_dimension_mean,
        CAST(0.2699 AS FLOAT64) AS radius_se, CAST(0.7886 AS FLOAT64) AS texture_se,
        CAST(2.058 AS FLOAT64) AS perimeter_se, CAST(23.56 AS FLOAT64) AS area_se,
        CAST(0.008462 AS FLOAT64) AS smoothness_se, CAST(0.0146 AS FLOAT64) AS compactness_se,
        CAST(0.02387 AS FLOAT64) AS concavity_se, CAST(0.01315 AS FLOAT64) AS concave_points_se,
        CAST(0.0198 AS FLOAT64) AS symmetry_se, CAST(0.0023 AS FLOAT64) AS fractal_dimension_se,
        CAST(15.11 AS FLOAT64) AS radius_worst, CAST(19.26 AS FLOAT64) AS texture_worst,
        CAST(99.70 AS FLOAT64) AS perimeter_worst, CAST(711.2 AS FLOAT64) AS area_worst,
        CAST(0.144 AS FLOAT64) AS smoothness_worst, CAST(0.1773 AS FLOAT64) AS compactness_worst,
        CAST(0.239 AS FLOAT64) AS concavity_worst, CAST(0.1288 AS FLOAT64) AS concave_points_worst,
        CAST(0.2977 AS FLOAT64) AS symmetry_worst, CAST(0.07259 AS FLOAT64) AS fractal_dimension_worst
      UNION ALL
      -- Sample 2: Malignant biopsy profile (large radius, irregular concavity)
      SELECT
        CAST(17.99 AS FLOAT64) AS radius_mean, CAST(10.38 AS FLOAT64) AS texture_mean,
        CAST(122.8 AS FLOAT64) AS perimeter_mean, CAST(1001.0 AS FLOAT64) AS area_mean,
        CAST(0.1184 AS FLOAT64) AS smoothness_mean, CAST(0.2776 AS FLOAT64) AS compactness_mean,
        CAST(0.3001 AS FLOAT64) AS concavity_mean, CAST(0.1471 AS FLOAT64) AS concave_points_mean,
        CAST(0.2419 AS FLOAT64) AS symmetry_mean, CAST(0.07871 AS FLOAT64) AS fractal_dimension_mean,
        CAST(1.095 AS FLOAT64) AS radius_se, CAST(0.9053 AS FLOAT64) AS texture_se,
        CAST(8.589 AS FLOAT64) AS perimeter_se, CAST(153.4 AS FLOAT64) AS area_se,
        CAST(0.006399 AS FLOAT64) AS smoothness_se, CAST(0.04904 AS FLOAT64) AS compactness_se,
        CAST(0.05373 AS FLOAT64) AS concavity_se, CAST(0.01587 AS FLOAT64) AS concave_points_se,
        CAST(0.03003 AS FLOAT64) AS symmetry_se, CAST(0.006193 AS FLOAT64) AS fractal_dimension_se,
        CAST(25.38 AS FLOAT64) AS radius_worst, CAST(17.33 AS FLOAT64) AS texture_worst,
        CAST(184.6 AS FLOAT64) AS perimeter_worst, CAST(2019.0 AS FLOAT64) AS area_worst,
        CAST(0.1622 AS FLOAT64) AS smoothness_worst, CAST(0.6656 AS FLOAT64) AS compactness_worst,
        CAST(0.7119 AS FLOAT64) AS concavity_worst, CAST(0.2654 AS FLOAT64) AS concave_points_worst,
        CAST(0.4601 AS FLOAT64) AS symmetry_worst, CAST(0.1189 AS FLOAT64) AS fractal_dimension_worst
    )
  );
```

---


- **Step 8**: Clean Up Resources (Optional)

```sql 
-- Cleanup resources
DROP MODEL IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.breast_cancer_model`;
DROP MODEL IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.breast_cancer_dnn_model`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_breast_cancer`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_breast_cancer`;
DROP SCHEMA IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox` CASCADE;
```

---
